A/B Testing & Hypothesis Testing Analysis

Objective: Perform statistical hypothesis testing on an A/B test dataset to determine if a new landing page or geographic location significantly impacts user conversion rates.

Tests Performed:

Independent T-Test: Compares the mean conversion rates between the Control and Treatment groups.

One-Way ANOVA: Compares the mean conversion rates across different countries (US, UK, CA).

Chi-Square Test: Tests the independence of the categorical variables (Group vs. Conversion).

In [1]:
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")

1. Load and Clean the Data

First, we load the datasets, filter out mismatched tracking data (where users in the control group mistakenly saw the new page, or vice-versa), remove duplicates, and merge the datasets on user_id.

In [2]:
# Load the datasets
ab_data = pd.read_csv('ab_data.csv')
countries = pd.read_csv('countries.csv')

# Filter for accurate tracking: group and landing_page must align
df = ab_data[((ab_data['group'] == 'treatment') == (ab_data['landing_page'] == 'new_page'))]

# Drop duplicates based on user_id
df = df.drop_duplicates(subset=['user_id'])

# Merge with countries dataset
df = df.merge(countries, on='user_id', how='inner')

print(f"Cleaned dataset shape: {df.shape}")
df.head()

Cleaned dataset shape: (290584, 6)


,user_id,timestamp,group,landing_page,converted,country
0,851104,2017-01-21 22:11:48.556739,control,old_page,0,US
1,804228,2017-01-12 08:01:45.159739,control,old_page,0,US
2,661590,2017-01-11 16:55:06.154213,treatment,new_page,0,US
3,853541,2017-01-08 18:28:03.143765,treatment,new_page,0,US
4,864975,2017-01-21 01:52:26.210827,control,old_page,1,US


2. Independent T-Test

Null Hypothesis (
): There is no difference in the mean conversion rate between the control group and the treatment group.
Alternative Hypothesis (
): There is a difference in the mean conversion rate between the two groups. (Note: While conversion is a binary metric, a T-test is mathematically valid here due to the extremely large sample size satisfying the Central Limit Theorem).

In [3]:
# Separate the conversion data by group
control_conv = df[df['group'] == 'control']['converted']
treatment_conv = df[df['group'] == 'treatment']['converted']

# Perform Independent 2-Sample T-test
t_stat, p_val_t = stats.ttest_ind(control_conv, treatment_conv, equal_var=False)

print(f"Control Conversion Rate: {control_conv.mean():.4f}")
print(f"Treatment Conversion Rate: {treatment_conv.mean():.4f}")
print("-" * 30)
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_val_t:.4f}")

Control Conversion Rate: 0.1204
Treatment Conversion Rate: 0.1188
------------------------------
T-Statistic: 1.3109
P-Value: 0.1899


3. One-Way ANOVA

Null Hypothesis (
): There is no difference in mean conversion rates across the three countries (US, UK, CA).
Alternative Hypothesis (
): At least one country has a significantly different mean conversion rate.

In [4]:
# Separate conversion data by country
us_conv = df[df['country'] == 'US']['converted']
uk_conv = df[df['country'] == 'UK']['converted']
ca_conv = df[df['country'] == 'CA']['converted']

# Perform One-Way ANOVA
f_stat, p_val_f = stats.f_oneway(us_conv, uk_conv, ca_conv)

print("Conversion Rates by Country:")
print(f"US: {us_conv.mean():.4f} | UK: {uk_conv.mean():.4f} | CA: {ca_conv.mean():.4f}")
print("-" * 30)
print(f"F-Statistic: {f_stat:.4f}")
print(f"P-Value: {p_val_f:.4f}")

Conversion Rates by Country:
US: 0.1195 | UK: 0.1206 | CA: 0.1153
------------------------------
F-Statistic: 1.6053
P-Value: 0.2008


4. Chi-Square Test of Independence

Null Hypothesis (
): The group (Control vs Treatment) and the converted status are independent (i.e., page version does not affect conversion).
Alternative Hypothesis (
): The group and converted status are dependent.

In [5]:
# Create a contingency table (cross-tabulation) of group vs. converted
contingency_table = pd.crosstab(df['group'], df['converted'])
display(contingency_table)

# Perform Chi-Square Test
chi2_stat, p_val_chi, dof, expected = stats.chi2_contingency(contingency_table)

print("-" * 30)
print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"Degrees of Freedom: {dof}")
print(f"P-Value: {p_val_chi:.4f}")

converted,0,1
group,,
control,127785,17489
treatment,128046,17264


------------------------------
Chi-Square Statistic: 1.7036
Degrees of Freedom: 1
P-Value: 0.1918



5. Summary of Findings

T-Test Results: The p-value (~0.190) is greater than our 0.05 threshold. We fail to reject the null hypothesis. The difference in conversion rates between the old page and the new page is not statistically significant.

ANOVA Results: The p-value (~0.201) is greater than 0.05. We fail to reject the null hypothesis. There is no statistically significant difference in conversion behavior across the US, UK, and Canada.

Chi-Square Results: The p-value (~0.192) is greater than 0.05. We fail to reject the null hypothesis. The conversion outcome is independent of which landing page the user was assigned to.

Conclusion: The new landing page does not drive significantly more conversions than the old page, overall or regionally. The business should retain the existing page rather than migrating to the new one.